# Breast Cancer 5-Year Survival — MICE for Both Numeric and Categorical

Numerical & categorical missing values are also imputed (not dropped).

**Strategy:** ordinal-encode categoricals (preserving NaN) → run MICE on the full numeric+ordinal matrix → round ordinal columns back to nearest valid category → OHE for the model.

Rows are only dropped if `Cohort` or `label` is missing.

**Numeric (8):** Age at Diagnosis, Nottingham prognostic index, gain, hetloss, Tumor Size, amp, TMB (nonsynonymous), homdel

**Categorical (4):** Type of Breast Surgery, Primary Tumor Laterality, Pam50 + Claudin-low subtype, Cellularity

**Models:** Logistic Regression + LightGBM

**Split rule:** Cohorts {4, 5} sealed as final holdout. CV on cohorts {1, 2, 3}, 3-fold leave-one-cohort-out.

In [1]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    classification_report,
)
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
HOLDOUT_COHORTS = {4.0, 5.0}

## Custom preprocessor: OrdinalEncode → MICE → Round → OHE

In [2]:
class FullMICEPreprocessor(BaseEstimator, TransformerMixin):
    """Imputes both numeric and categorical missing values with MICE.

    Steps:
    1. Ordinal-encode categoricals (NaN stays NaN via encoded_missing_value).
    2. Stack numeric + ordinal into one matrix.
    3. MICE on the full matrix (fitted on training data only).
    4. Round ordinal columns back to nearest valid integer category.
    5. OHE the rounded categoricals for the model.
    6. Optionally StandardScale the numeric columns (for LR).
    """

    def __init__(self, numeric_cols, categorical_cols,
                 scale_numeric=True, random_state=42, max_iter=10):
        self.numeric_cols     = numeric_cols
        self.categorical_cols = categorical_cols
        self.scale_numeric    = scale_numeric
        self.random_state     = random_state
        self.max_iter         = max_iter

    def fit(self, X, y=None):
        self.ordinal_ = OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=np.nan,
            encoded_missing_value=np.nan,
        )
        self.ordinal_.fit(X[self.categorical_cols])
        self.categories_ = self.ordinal_.categories_

        num  = X[self.numeric_cols].to_numpy(dtype=float)
        cat  = self.ordinal_.transform(X[self.categorical_cols])
        full = np.hstack([num, cat])
        self.n_num_ = num.shape[1]

        self.imputer_ = IterativeImputer(
            random_state=self.random_state, max_iter=self.max_iter
        )
        self.imputer_.fit(full)

        cat_nonan = X[self.categorical_cols].dropna()
        self.ohe_ = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.ohe_.fit(cat_nonan)

        if self.scale_numeric:
            imputed_num = self.imputer_.transform(full)[:, :self.n_num_]
            self.scaler_ = StandardScaler()
            self.scaler_.fit(imputed_num)

        return self

    def transform(self, X):
        num  = X[self.numeric_cols].to_numpy(dtype=float)
        cat  = self.ordinal_.transform(X[self.categorical_cols])
        full = np.hstack([num, cat])

        imputed     = self.imputer_.transform(full)
        imputed_num = imputed[:, :self.n_num_]
        imputed_cat = imputed[:, self.n_num_:]

        imputed_cat = np.round(imputed_cat).astype(int)
        for i, cats in enumerate(self.categories_):
            imputed_cat[:, i] = np.clip(imputed_cat[:, i], 0, len(cats) - 1)

        cat_labels = self.ordinal_.inverse_transform(imputed_cat.astype(float))
        # wrap in DataFrame so OHE gets the column names it was fitted with
        cat_labels_df = pd.DataFrame(cat_labels, columns=self.categorical_cols)
        cat_ohe = self.ohe_.transform(cat_labels_df)

        if self.scale_numeric:
            imputed_num = self.scaler_.transform(imputed_num)

        return np.hstack([imputed_num, cat_ohe])

## 1. Load data and build the 5-year survival label

| Condition | Label |
|---|---|
| DECEASED and OS_MONTHS < 60 | 0 — died before 5 years |
| followed >= 60 months | 1 — survived to 5 years |
| LIVING and OS_MONTHS < 60 | excluded — censored, true outcome unknown |

In [3]:
df = pd.read_csv('../data/Final_metabric_data.tsv', sep='\t')

print(f'Raw data shape: {df.shape}')
print(df.head().to_string())

died_early  = (df['Overall Survival Status'] == '1:DECEASED') & (df['Overall Survival (Months)'] < 60)
survived_5y = df['Overall Survival (Months)'] >= 60
censored    = (df['Overall Survival Status'] == '0:LIVING')  & (df['Overall Survival (Months)'] < 60)

df['label'] = np.nan
df.loc[died_early,  'label'] = 0
df.loc[survived_5y, 'label'] = 1
df = df[~censored].reset_index(drop=True)

print(f'\nRows after excluding censored: {len(df)}')
print(df['label'].value_counts().rename({0: 'died <5yr', 1: 'survived 5yr'}))

Raw data shape: (1981, 30)
  Patient ID  TMB (nonsynonymous) Oncotree Code  Age at Diagnosis  Cohort ER Status  Tumor Size  Nottingham prognostic index Radio Therapy Inferred Menopausal State Integrative Cluster Hormone Therapy HER2 Status HER2 status measured by SNP6 Pam50 + Claudin-low subtype Chemotherapy Type of Breast Surgery Cellularity Primary Tumor Laterality  homdel  hetloss    gain    amp Relapse Free Status  Relapse Free Status (Months)  Relapse Free Status (Years) Overall Survival Status  Overall Survival (Months)  Overall Survival (Years) Patient's Vital Status
0    MB-0000             0.000000           IDC             75.65     1.0  Positive        22.0                        6.044           YES                      Post                4ER+             YES    Negative                      NEUTRAL                 claudin-low           NO             MASTECTOMY         NaN                    Right   0.000    0.018   0.127  0.000      0:Not Recurred                    140.5

## 2. Select features — drop only where Cohort or label is missing

In [4]:
NUMERIC = [
    'Age at Diagnosis',
    'Nottingham prognostic index',
    'gain',
    'hetloss',
    'Tumor Size',
    'amp',
    'TMB (nonsynonymous)',
    'homdel',
]

CATEGORICAL = [
    'Type of Breast Surgery',
    'Primary Tumor Laterality',
    'Pam50 + Claudin-low subtype',
    'Cellularity',
]

FEATURES = NUMERIC + CATEGORICAL

subset = df[FEATURES + ['Cohort', 'label']]
print(f'Rows before dropna: {len(subset)}')
print('Missing values per column:')
print(subset.isna().sum().to_string())
print(f'Rows with at least one missing value: {subset.isna().any(axis=1).sum()}')

# Only drop where Cohort or label is missing — all feature NaNs handled by MICE
df = subset.dropna(subset=['Cohort', 'label']).reset_index(drop=True)
print(f'\nRows after dropna (Cohort/label only): {len(df)} (dropped {len(subset) - len(df)} rows)')

X      = df[FEATURES]
y      = df['label'].astype(int)
groups = df['Cohort']

print(f'Numeric missing values (imputed by MICE): {X[NUMERIC].isna().sum().sum()}')
print(f'Categorical missing values (imputed by MICE): {X[CATEGORICAL].isna().sum().sum()}')
print(f'Label distribution: {y.value_counts().to_dict()}')
print(f'Cohorts present: {sorted(groups.unique())}')
print('\nFirst 5 rows of feature matrix:')
print(X.head().to_string())

Rows before dropna: 1917
Missing values per column:
Age at Diagnosis                 0
Nottingham prognostic index      0
gain                             0
hetloss                          0
Tumor Size                      24
amp                              0
TMB (nonsynonymous)              0
homdel                           0
Type of Breast Surgery          20
Primary Tumor Laterality       105
Pam50 + Claudin-low subtype      1
Cellularity                     63
Cohort                           0
label                            0
Rows with at least one missing value: 191

Rows after dropna (Cohort/label only): 1917 (dropped 0 rows)
Numeric missing values (imputed by MICE): 24
Categorical missing values (imputed by MICE): 189
Label distribution: {1: 1490, 0: 427}
Cohorts present: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

First 5 rows of feature matrix:
   Age at Diagnosis  Nottingham prognostic index    gain  hetloss  Tumor Size    amp 

## 3. Split: seal cohorts {4, 5} as final holdout

In [5]:
holdout_mask = groups.isin(HOLDOUT_COHORTS)

X_dev,  y_dev,  groups_dev = (X[~holdout_mask].reset_index(drop=True),
                               y[~holdout_mask].reset_index(drop=True),
                               groups[~holdout_mask].reset_index(drop=True))
X_hold, y_hold             = (X[holdout_mask].reset_index(drop=True),
                               y[holdout_mask].reset_index(drop=True))

print(f'Dev pool : {len(X_dev)} rows, cohorts {sorted(groups_dev.unique())}')
print(f'Holdout  : {len(X_hold)} rows, cohorts {sorted(groups[holdout_mask].unique())} — sealed until Section 6')

Dev pool : 1518 rows, cohorts [np.float64(1.0), np.float64(2.0), np.float64(3.0)]
Holdout  : 399 rows, cohorts [np.float64(4.0), np.float64(5.0)] — sealed until Section 6


## 4. Model pipelines

In [6]:
spw = (y_dev == 0).sum() / (y_dev == 1).sum()

MODELS = {
    'logistic_regression': Pipeline([
        ('prep',  FullMICEPreprocessor(NUMERIC, CATEGORICAL, scale_numeric=True,  random_state=RANDOM_STATE)),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'lightgbm': Pipeline([
        ('prep',  FullMICEPreprocessor(NUMERIC, CATEGORICAL, scale_numeric=False, random_state=RANDOM_STATE)),
        ('model', LGBMClassifier(scale_pos_weight=spw, random_state=RANDOM_STATE, verbosity=-1)),
    ]),
}

## 5. Leave-one-cohort-out CV on the dev pool (cohorts 1, 2, 3)

3 folds: each fold trains on 2 cohorts and validates on the entire cohort left out.

In [7]:
cv_rows = []
gkf = GroupKFold(n_splits=groups_dev.nunique())  # n_splits = 3

for model_name, pipe in MODELS.items():
    print(f'Running CV for {model_name}...')
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_dev, y_dev, groups=groups_dev)):
        fold_pipe = clone(pipe)
        fold_pipe.fit(X_dev.iloc[train_idx], y_dev.iloc[train_idx])

        proba = fold_pipe.predict_proba(X_dev.iloc[val_idx])[:, 1]
        preds = fold_pipe.predict(X_dev.iloc[val_idx])
        val_cohort = groups_dev.iloc[val_idx].iloc[0]

        cv_rows.append({
            'model':      model_name,
            'fold':       fold,
            'val_cohort': val_cohort,
            'n_train':    len(train_idx),
            'n_val':      len(val_idx),
            'roc_auc':    roc_auc_score(y_dev.iloc[val_idx], proba),
            'pr_auc':     average_precision_score(y_dev.iloc[val_idx], proba),
            'precision':  precision_score(y_dev.iloc[val_idx], preds),
            'recall':     recall_score(y_dev.iloc[val_idx], preds),
            'f1':         f1_score(y_dev.iloc[val_idx], preds),
        })

cv = pd.DataFrame(cv_rows)
print('\n--- Per-fold CV results ---')
print(cv.to_string(index=False))

print('\n--- CV summary (mean ± std across 3 folds) ---')
summary = cv.groupby('model')[['roc_auc', 'pr_auc', 'precision', 'recall', 'f1']].agg(['mean', 'std']).round(3)
print(summary.to_string())

Running CV for logistic_regression...
Running CV for lightgbm...

--- Per-fold CV results ---
              model  fold  val_cohort  n_train  n_val  roc_auc   pr_auc  precision   recall       f1
logistic_regression     0         3.0      768    750 0.699006 0.886850   0.859688 0.663230 0.748788
logistic_regression     1         1.0     1038    480 0.774103 0.905952   0.896679 0.656757 0.758190
logistic_regression     2         2.0     1230    288 0.745142 0.906584   0.887255 0.776824 0.828375
           lightgbm     0         3.0      768    750 0.663793 0.862670   0.813480 0.891753 0.850820
           lightgbm     1         1.0     1038    480 0.743636 0.906119   0.828804 0.824324 0.826558
           lightgbm     2         2.0     1230    288 0.723917 0.911430   0.847826 0.836910 0.842333

--- CV summary (mean ± std across 3 folds) ---
                    roc_auc        pr_auc        precision        recall            f1       
                       mean    std   mean    std      mea

## 6. Final evaluation on the holdout (cohorts 4, 5) — run once only

In [8]:
for model_name, pipe in MODELS.items():
    pipe.fit(X_dev, y_dev)
    proba = pipe.predict_proba(X_hold)[:, 1]
    preds = pipe.predict(X_hold)

    print(f'\n=== {model_name} — holdout (cohorts 4, 5) ===')
    print(f'  ROC-AUC : {roc_auc_score(y_hold, proba):.3f}')
    print(f'  PR-AUC  : {average_precision_score(y_hold, proba):.3f}')
    print(classification_report(y_hold, preds, target_names=['died_<5y', 'survived_5y']))


=== logistic_regression — holdout (cohorts 4, 5) ===
  ROC-AUC : 0.751
  PR-AUC  : 0.890
              precision    recall  f1-score   support

    died_<5y       0.38      0.76      0.51        94
 survived_5y       0.89      0.62      0.73       305

    accuracy                           0.65       399
   macro avg       0.64      0.69      0.62       399
weighted avg       0.77      0.65      0.68       399


=== lightgbm — holdout (cohorts 4, 5) ===
  ROC-AUC : 0.732
  PR-AUC  : 0.888
              precision    recall  f1-score   support

    died_<5y       0.46      0.49      0.48        94
 survived_5y       0.84      0.83      0.83       305

    accuracy                           0.75       399
   macro avg       0.65      0.66      0.65       399
weighted avg       0.75      0.75      0.75       399

